# Retrieval-Augmented Generation (RAG)

In [ ]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

Create a static calorie table that we can use as a tool:

In [39]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")
nutrition_qna_db = chroma_client.get_collection(name="nutrition_qna")

In [40]:
results = nutrition_qna_db.query(query_texts=["explain malnutrition during pregnancy"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', True)]
Question: What are the implications of not consuming adequate nutrients during pregnancy on a fetus?
        Answer: Malnutrition during pregnancy can have severe repercussions for an unborn child. It might lead to early delivery and, if sustained, could result in a baby with low birth weight. This often results in developmental issues, poor growth, and heightened susceptibility to diseases.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', True)]
Question: What issues arise due to insufficient nourishment while expecting?
        Answer: Malnutrition during pregnancy can lead to premature delivery or low birth weight. It is essential for pregnant women to maintain a balanced diet to ensure proper growth and development of the fetus.

        This Q&A pair provides information about nutrition and health topics.




In [46]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

@function_tool
def nutrition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function too ask a question about nutrition.

    Args:
        query: The question to ask
        max_results: The maximum number of results to return.

    Returns:
        A string containing the question and the answer related to the query.
    """

    results = nutrition_qna_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [48]:
# calorie_lookup_tool('bananas', 4)
# nutrition_qna_tool('Explain malnutrition during pregnancy', 2)


In [51]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and nutrition advise.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool. If you need to look up nutrition information, use the nutrition_qna_tool
    """,
    tools= [calorie_lookup_tool, nutrition_qna_tool]
)

In [55]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in whiskey and a pizza? Also give calories per 100g. Are they good to consume during pregnancy",
    )
    print(result.final_output)

Here are general estimates (varies by exact type and portion):

- Whiskey
  - 1 standard drink (about 44 ml, 40% ABV): ~97 kcal
  - Per 100 ml: ~220–250 kcal

- Pizza
  - Per 100 g: ~250–300 kcal (depends on crust, cheese, toppings)
  - Whole pizza kcal depends on size and toppings (often 800–1,200 kcal+)

Pregnancy guidance:
- Alcohol: avoid completely. No safe level has been established.
- Pizza: okay as part of a balanced diet if chosen with good ingredients and cooked safely; watch for high sodium and saturated fat, and ensure it’s well heated.

If you provide exact pizza type and serving size, I can give a more precise total and per-100g calorie count.
